In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

In [2]:

fake = Faker()
num_customers = 150
transactions_per_customer = 150
customers = []
for _ in range(num_customers):
    first_name = fake.first_name()
    last_name = fake.last_name()
    customer_id = fake.unique.uuid4()  # Unique customer ID
    r = random.random()
    if r < 0.33:
        customer_type = "frequent"
    elif r < 0.66:
        customer_type = "regular"
    else:
        customer_type = "rare"
    customers.append([first_name, last_name, customer_id, customer_type])

In [3]:
df_customers = pd.DataFrame(
    customers,
    columns=['first_name', 'last_name', 'customer_id', 'customer_type']
)
transactions = []
vendors = ['Walmart', 'Amazon', 'Costco', 'Starbucks', 'American Airlines','United Airlines','CVS Pharmacy','Delta Airlines','Subway','Chick-Fil-A',
           'Chipotle','Panera Bread','Macdonalds']

# Vendor list (for retail, food chains, airlines, and pharmacy)
retail_and_airlines_vendors = ['Walmart', 'Amazon', 'Costco', 'American Airlines', 'Delta Airlines', 'United Airlines']
food_chains = ['Starbucks', 'Chipotle', 'McDonald\'s', 'Panera Bread', 'Subway', 'Chick-Fil-A']
pharmacy = ['CVS Pharmacy']

vendor_groups = {
    0: ['American Airlines', 'Walmart'],  # First 30 customers
    1: ['Delta Airlines', 'Costco'],  # Next 30 customers
    2: ['United Airlines', 'Amazon'],  # Next 30 customers
    3: ['American Airlines', 'Costco'],  # Next 30 customers
    4: ['Walmart', 'Delta Airlines'],  # Last 30 customers
}

In [4]:
print(df_customers.head())
print(df_customers.columns)
print(df_customers['customer_type'].value_counts())

  first_name last_name                           customer_id customer_type
0        Kim    Arnold  8d375321-70c4-48dd-8fcf-a762798f8ca7          rare
1  Christine    Jordan  39ddb050-c16b-4f09-ae03-d2b29c7a759f       regular
2       Jose    Gibson  e3f9058d-06bc-40cf-b739-6b13d0b7fe3f       regular
3    Matthew    Garner  5d6ef36e-90c3-4dcf-a854-7d8dc2015145       regular
4      Megan  Hamilton  5313a2e0-0d3b-447d-8e6c-e6db37f4b0a4          rare
Index(['first_name', 'last_name', 'customer_id', 'customer_type'], dtype='str')
customer_type
frequent    53
regular     49
rare        48
Name: count, dtype: int64


In [5]:

# Function to generate transaction amounts based on vendor
def generate_transaction_amount(vendor):
    if vendor in ['American Airlines', 'United Airlines', 'Delta Airlines']:
        return round(random.uniform(100.0, 500.0), 2)  # Airline vendors between $100 and $500
    elif vendor in ['Walmart', 'Costco', 'Amazon']:
        return round(random.uniform(50.0, 500.0), 2)  # Retail vendors between $50 and $500
    elif vendor in ['Starbucks', 'Chipotle', 'McDonald\'s', 'Panera Bread', 'Subway', 'Chick-Fil-A']:
        return round(random.uniform(5.0, 100.0), 2)  # Food chains between $5 and $100
    elif vendor == 'CVS Pharmacy':
        return round(random.uniform(10.0, 100.0), 2)  # CVS Pharmacy between $10 and $100
    else:
        return round(random.uniform(80.0, 500.0), 2)  # Other vendors between $80 and $500


for idx, customer in df_customers.iterrows():
    first_name = customer['first_name']
    last_name = customer['last_name']
    customer_id = customer['customer_id']
    customer_type = customer['customer_type']

    # Random starting date for the first transaction
    start_date = datetime(2024, 1, 1)
    end_date = datetime(2025, 12, 31)

    # Determine primary and secondary vendors for the customer based on their index
    primary_vendors = vendor_groups[idx // 30]  # Determine primary vendors based on index
    secondary_vendors = [vendor for vendor in vendors if vendor not in primary_vendors]

    # Create a transaction list for each customer
    all_vendors = primary_vendors + secondary_vendors

    for i in range(transactions_per_customer):
        # Select a vendor based on primary and secondary
        vendor = random.choice(all_vendors)

        # Base days between transactions by customer_type: frequent 3-7, regular 8-15, rare 16-30
        if customer_type == 'frequent':
            base_days = random.randint(3, 7)
        elif customer_type == 'regular':
            base_days = random.randint(8, 15)
        else:
            base_days = random.randint(16, 30)

        # Vendor adjustment
        if vendor in ['American Airlines', 'United Airlines', 'Delta Airlines']:
            base_days += random.randint(5, 10)
        elif vendor in ['Walmart', 'Amazon', 'Costco']:
            base_days += random.randint(-3, 3)
        elif vendor in ['Starbucks', 'Chipotle', 'McDonald\'s', 'Panera Bread', 'Subway', 'Chick-Fil-A', 'Macdonalds']:
            base_days += random.randint(-2, 2)
        elif vendor == 'CVS Pharmacy':
            base_days += random.randint(-5, 5)

        # Add noise
        base_days += random.randint(-2, 2)
        days_diff = max(0, base_days)

        transaction_date = start_date + timedelta(days=i * days_diff)
        if transaction_date > end_date:
            break

        # Generate a random transaction amount based on the vendor
        amount = generate_transaction_amount(vendor)

        # Randomly assign "Debit" or "Credit" transaction type
        transaction_type = random.choice(['Debit', 'Credit'])

        # Add transaction to the list
        transactions.append([first_name, last_name, customer_id, fake.uuid4(), transaction_date, amount, vendor, transaction_type])

# Create DataFrame from the transactions list
df_transactions = pd.DataFrame(transactions, columns=['first_name', 'last_name', 'customer_id', 'transaction_id', 'transaction_datetime', 
                                                      'amount', 'vendor', 'transaction_type'])

# View the first few transactions to verify vendor distribution
print(df_transactions[['customer_id', 'vendor', 'transaction_datetime']].head(20))





                             customer_id             vendor  \
0   8d375321-70c4-48dd-8fcf-a762798f8ca7        Chick-Fil-A   
1   8d375321-70c4-48dd-8fcf-a762798f8ca7       Panera Bread   
2   8d375321-70c4-48dd-8fcf-a762798f8ca7             Amazon   
3   8d375321-70c4-48dd-8fcf-a762798f8ca7     Delta Airlines   
4   8d375321-70c4-48dd-8fcf-a762798f8ca7  American Airlines   
5   8d375321-70c4-48dd-8fcf-a762798f8ca7        Chick-Fil-A   
6   8d375321-70c4-48dd-8fcf-a762798f8ca7             Amazon   
7   8d375321-70c4-48dd-8fcf-a762798f8ca7       Panera Bread   
8   8d375321-70c4-48dd-8fcf-a762798f8ca7          Starbucks   
9   8d375321-70c4-48dd-8fcf-a762798f8ca7       CVS Pharmacy   
10  8d375321-70c4-48dd-8fcf-a762798f8ca7       CVS Pharmacy   
11  8d375321-70c4-48dd-8fcf-a762798f8ca7       Panera Bread   
12  8d375321-70c4-48dd-8fcf-a762798f8ca7        Chick-Fil-A   
13  8d375321-70c4-48dd-8fcf-a762798f8ca7     Delta Airlines   
14  8d375321-70c4-48dd-8fcf-a762798f8ca7             Su

In [6]:
# Calculate "likelihood prediction" (time difference between consecutive transactions per customer)
df_transactions['next_transaction_date'] = df_transactions.groupby(['customer_id'])['transaction_datetime'].shift(-1)

# Calculate the difference in days between current and next transaction date
df_transactions['likelihood_prediction'] = (df_transactions['next_transaction_date'] - df_transactions['transaction_datetime']).dt.days

# For same-day transactions (same vendor and same date), set likelihood to 0
df_transactions['likelihood_prediction'] = df_transactions['likelihood_prediction'].fillna(0)

print("Min gap:", df_transactions['likelihood_prediction'].min())
print("Max gap:", df_transactions['likelihood_prediction'].max())
print("Mean gap:", df_transactions['likelihood_prediction'].mean())
print(df_transactions['likelihood_prediction'].describe())

# Display the first few rows with likelihood prediction
print(df_transactions[['customer_id', 'vendor', 'transaction_datetime', 'likelihood_prediction']].head(10))

Min gap: -700.0
Max gap: 658.0
Mean gap: 10.67887184562098
count    6063.000000
mean       10.678872
std       144.547010
min      -700.000000
25%       -53.000000
50%         9.000000
75%        75.000000
max       658.000000
Name: likelihood_prediction, dtype: float64
                            customer_id             vendor  \
0  8d375321-70c4-48dd-8fcf-a762798f8ca7        Chick-Fil-A   
1  8d375321-70c4-48dd-8fcf-a762798f8ca7       Panera Bread   
2  8d375321-70c4-48dd-8fcf-a762798f8ca7             Amazon   
3  8d375321-70c4-48dd-8fcf-a762798f8ca7     Delta Airlines   
4  8d375321-70c4-48dd-8fcf-a762798f8ca7  American Airlines   
5  8d375321-70c4-48dd-8fcf-a762798f8ca7        Chick-Fil-A   
6  8d375321-70c4-48dd-8fcf-a762798f8ca7             Amazon   
7  8d375321-70c4-48dd-8fcf-a762798f8ca7       Panera Bread   
8  8d375321-70c4-48dd-8fcf-a762798f8ca7          Starbucks   
9  8d375321-70c4-48dd-8fcf-a762798f8ca7       CVS Pharmacy   

  transaction_datetime  likelihood_prediction 

In [7]:
# Recompute next transaction date
df_transactions = df_transactions.sort_values(
    ['customer_id', 'transaction_datetime']
)

df_transactions['next_transaction_date'] = (
    df_transactions.groupby(['customer_id'])['transaction_datetime']
    .shift(-1)
)

df_transactions['likelihood_prediction'] = (
    df_transactions['next_transaction_date'] - df_transactions['transaction_datetime']
).dt.days

# Drop last transactions (no next date)
df_transactions = df_transactions.dropna(subset=['next_transaction_date'])

# Basic checks
print("Shape:", df_transactions.shape)
print("Date range:", df_transactions['transaction_datetime'].min(), "to", df_transactions['transaction_datetime'].max())
print("Min days:", df_transactions['likelihood_prediction'].min())
print("Max days:", df_transactions['likelihood_prediction'].max())
print("Mean days:", df_transactions['likelihood_prediction'].mean())

Shape: (5913, 10)
Date range: 2024-01-01 00:00:00 to 2025-12-15 00:00:00
Min days: 0.0
Max days: 262.0
Mean days: 15.841028242854726


In [8]:
# Number of transactions per customer
df_transactions['customer_transaction_count'] = (
    df_transactions.groupby('customer_id')['transaction_id']
    .transform('count')
)

print(df_transactions[['customer_id', 'customer_transaction_count']].head(10))

                               customer_id  customer_transaction_count
5613  009a4176-56d5-44bf-b029-3ff859884c82                          26
5614  009a4176-56d5-44bf-b029-3ff859884c82                          26
5615  009a4176-56d5-44bf-b029-3ff859884c82                          26
5618  009a4176-56d5-44bf-b029-3ff859884c82                          26
5616  009a4176-56d5-44bf-b029-3ff859884c82                          26
5617  009a4176-56d5-44bf-b029-3ff859884c82                          26
5619  009a4176-56d5-44bf-b029-3ff859884c82                          26
5620  009a4176-56d5-44bf-b029-3ff859884c82                          26
5624  009a4176-56d5-44bf-b029-3ff859884c82                          26
5622  009a4176-56d5-44bf-b029-3ff859884c82                          26


In [16]:
# How many times a customer purchased from a vendor
df_transactions['customer_vendor_txn_count'] = (
    df_transactions.groupby(['customer_id', 'vendor'])['transaction_id']
    .transform('count')
)

print(df_transactions[['customer_id', 'vendor', 'customer_vendor_txn_count']].head(10))

                            customer_id             vendor  \
0  009a4176-56d5-44bf-b029-3ff859884c82             Amazon   
1  009a4176-56d5-44bf-b029-3ff859884c82  American Airlines   
2  009a4176-56d5-44bf-b029-3ff859884c82  American Airlines   
3  009a4176-56d5-44bf-b029-3ff859884c82  American Airlines   
4  009a4176-56d5-44bf-b029-3ff859884c82       CVS Pharmacy   
5  009a4176-56d5-44bf-b029-3ff859884c82       CVS Pharmacy   
6  009a4176-56d5-44bf-b029-3ff859884c82        Chick-Fil-A   
7  009a4176-56d5-44bf-b029-3ff859884c82           Chipotle   
8  009a4176-56d5-44bf-b029-3ff859884c82             Costco   
9  009a4176-56d5-44bf-b029-3ff859884c82             Costco   

   customer_vendor_txn_count  
0                          1  
1                          3  
2                          3  
3                          3  
4                          2  
5                          2  
6                          1  
7                          1  
8                          3  
9      

In [17]:
# Days since customer's first transaction
df_transactions['days_since_first_purchase'] = (
    df_transactions['transaction_datetime'] -
    df_transactions.groupby('customer_id')['transaction_datetime'].transform('min')
).dt.days

print(df_transactions[['customer_id', 'days_since_first_purchase']].head(10))

                            customer_id  days_since_first_purchase
0  009a4176-56d5-44bf-b029-3ff859884c82                        126
1  009a4176-56d5-44bf-b029-3ff859884c82                          0
2  009a4176-56d5-44bf-b029-3ff859884c82                         64
3  009a4176-56d5-44bf-b029-3ff859884c82                        396
4  009a4176-56d5-44bf-b029-3ff859884c82                        464
5  009a4176-56d5-44bf-b029-3ff859884c82                        540
6  009a4176-56d5-44bf-b029-3ff859884c82                        260
7  009a4176-56d5-44bf-b029-3ff859884c82                         93
8  009a4176-56d5-44bf-b029-3ff859884c82                         80
9  009a4176-56d5-44bf-b029-3ff859884c82                        176


In [18]:
# Month and weekday features
df_transactions['transaction_month'] = df_transactions['transaction_datetime'].dt.month
df_transactions['transaction_weekday'] = df_transactions['transaction_datetime'].dt.weekday

print(df_transactions[['transaction_datetime', 'transaction_month', 'transaction_weekday']].head(10))

  transaction_datetime  transaction_month  transaction_weekday
0           2024-05-06                  5                    0
1           2024-01-01                  1                    0
2           2024-03-05                  3                    1
3           2025-01-31                  1                    4
4           2025-04-09                  4                    2
5           2025-06-24                  6                    1
6           2024-09-17                  9                    1
7           2024-04-03                  4                    2
8           2024-03-21                  3                    3
9           2024-06-25                  6                    1


In [19]:
# Average spending per customer
df_transactions['customer_avg_spend'] = (
    df_transactions.groupby('customer_id')['amount']
    .transform('mean')
)

print(df_transactions[['customer_id', 'amount', 'customer_avg_spend']].head(10))

                            customer_id  amount  customer_avg_spend
0  009a4176-56d5-44bf-b029-3ff859884c82  112.53          193.128077
1  009a4176-56d5-44bf-b029-3ff859884c82  279.22          193.128077
2  009a4176-56d5-44bf-b029-3ff859884c82  334.06          193.128077
3  009a4176-56d5-44bf-b029-3ff859884c82  190.38          193.128077
4  009a4176-56d5-44bf-b029-3ff859884c82   74.35          193.128077
5  009a4176-56d5-44bf-b029-3ff859884c82   86.65          193.128077
6  009a4176-56d5-44bf-b029-3ff859884c82   93.98          193.128077
7  009a4176-56d5-44bf-b029-3ff859884c82   31.70          193.128077
8  009a4176-56d5-44bf-b029-3ff859884c82  288.90          193.128077
9  009a4176-56d5-44bf-b029-3ff859884c82  250.67          193.128077


In [20]:
# Average time gap between purchases for each customer
df_transactions['customer_avg_gap'] = (
    df_transactions.groupby('customer_id')['likelihood_prediction']
    .transform('mean')
)

print(df_transactions[['customer_id','likelihood_prediction','customer_avg_gap']].head(10))

                            customer_id  likelihood_prediction  \
0  009a4176-56d5-44bf-b029-3ff859884c82                    NaN   
1  009a4176-56d5-44bf-b029-3ff859884c82                   64.0   
2  009a4176-56d5-44bf-b029-3ff859884c82                  332.0   
3  009a4176-56d5-44bf-b029-3ff859884c82                    NaN   
4  009a4176-56d5-44bf-b029-3ff859884c82                   76.0   
5  009a4176-56d5-44bf-b029-3ff859884c82                    NaN   
6  009a4176-56d5-44bf-b029-3ff859884c82                    NaN   
7  009a4176-56d5-44bf-b029-3ff859884c82                    NaN   
8  009a4176-56d5-44bf-b029-3ff859884c82                   96.0   
9  009a4176-56d5-44bf-b029-3ff859884c82                   97.0   

   customer_avg_gap  
0        132.142857  
1        132.142857  
2        132.142857  
3        132.142857  
4        132.142857  
5        132.142857  
6        132.142857  
7        132.142857  
8        132.142857  
9        132.142857  


In [21]:
# Remove rows where likelihood_prediction is NaN
df_transactions = df_transactions.dropna(subset=['likelihood_prediction'])

print("Dataset shape after cleaning:", df_transactions.shape)

Dataset shape after cleaning: (4100, 17)


In [22]:
# Features for model
features = [
    'amount',
    'customer_transaction_count',
    'customer_vendor_txn_count',
    'days_since_first_purchase',
    'transaction_month',
    'transaction_weekday',
    'customer_avg_spend',
    'customer_avg_gap'
]

X = df_transactions[features]
y = df_transactions['likelihood_prediction']

print(X.head())
print("Target sample:")
print(y.head())

   amount  customer_transaction_count  customer_vendor_txn_count  \
1  279.22                          26                          3   
2  334.06                          26                          3   
4   74.35                          26                          2   
8  288.90                          26                          3   
9  250.67                          26                          3   

   days_since_first_purchase  transaction_month  transaction_weekday  \
1                          0                  1                    0   
2                         64                  3                    1   
4                        464                  4                    2   
8                         80                  3                    3   
9                        176                  6                    1   

   customer_avg_spend  customer_avg_gap  
1          193.128077        132.142857  
2          193.128077        132.142857  
4          193.128077        132

In [9]:
print("Shape:", df_transactions.shape)
print("Date range:", 
      df_transactions['transaction_datetime'].min(), 
      "to", 
      df_transactions['transaction_datetime'].max())

print(df_transactions['likelihood_prediction'].describe())

Shape: (5913, 11)
Date range: 2024-01-01 00:00:00 to 2025-12-15 00:00:00
count    5913.000000
mean       15.841028
std        21.167275
min         0.000000
25%         3.000000
50%         9.000000
75%        20.000000
max       262.000000
Name: likelihood_prediction, dtype: float64


In [10]:
print("Date range:", df_transactions['transaction_datetime'].min(), "to", df_transactions['transaction_datetime'].max())
print("Min likelihood:", df_transactions['likelihood_prediction'].min())
print("Max likelihood:", df_transactions['likelihood_prediction'].max())
print("Any negative?", (df_transactions['likelihood_prediction'] < 0).any())

Date range: 2024-01-01 00:00:00 to 2025-12-15 00:00:00
Min likelihood: 0.0
Max likelihood: 262.0
Any negative? False


In [11]:
print(df_transactions['likelihood_prediction'].describe())
print(df_transactions.groupby('customer_id')['likelihood_prediction'].mean().head())
print(df_transactions.groupby('vendor')['likelihood_prediction'].mean())

count    5913.000000
mean       15.841028
std        21.167275
min         0.000000
25%         3.000000
50%         9.000000
75%        20.000000
max       262.000000
Name: likelihood_prediction, dtype: float64
customer_id
009a4176-56d5-44bf-b029-3ff859884c82    27.038462
021c1674-83f2-4370-adc1-5b416f068bc0    10.526316
0340ef88-edcf-4eb2-882a-1ec20b0179a7    14.000000
040d04c2-7a52-486f-85f0-77281daee7ff    18.000000
0470d9bb-6d59-48ab-8046-1babfc0e5ee8    17.000000
Name: likelihood_prediction, dtype: float64
vendor
Amazon               14.002101
American Airlines    23.389034
CVS Pharmacy         15.889362
Chick-Fil-A          13.965517
Chipotle             14.654348
Costco               12.984055
Delta Airlines       20.170157
Macdonalds           16.030738
Panera Bread         15.176842
Starbucks            14.602041
Subway               13.618449
United Airlines      19.357143
Walmart              14.456842
Name: likelihood_prediction, dtype: float64


In [12]:
same_day_transactions = df_transactions[df_transactions.duplicated(
    subset=['customer_id', 'vendor', 'transaction_datetime'], keep=False
)]
print(same_day_transactions[['customer_id', 'vendor', 'transaction_datetime']].head(10))

                               customer_id        vendor transaction_datetime
1509  0340ef88-edcf-4eb2-882a-1ec20b0179a7        Costco           2024-01-01
1525  0340ef88-edcf-4eb2-882a-1ec20b0179a7  CVS Pharmacy           2024-01-01
1536  0340ef88-edcf-4eb2-882a-1ec20b0179a7  CVS Pharmacy           2024-01-01
1539  0340ef88-edcf-4eb2-882a-1ec20b0179a7        Costco           2024-01-01
3130  07ed0d03-feaa-4021-a431-54bac3baab4b  CVS Pharmacy           2024-01-01
3147  07ed0d03-feaa-4021-a431-54bac3baab4b  CVS Pharmacy           2024-01-01
3006  141c9ade-9b32-4cc3-adee-bb3624d0c39f   Chick-Fil-A           2025-03-08
3030  141c9ade-9b32-4cc3-adee-bb3624d0c39f   Chick-Fil-A           2025-03-08
792   166f25d7-3591-4955-b3dd-12fc5abc2dcf       Walmart           2024-01-01
855   166f25d7-3591-4955-b3dd-12fc5abc2dcf       Walmart           2024-01-01


In [13]:
# Recompute next_transaction_date and likelihood per customer+vendor

df_transactions = df_transactions.sort_values(
    by=['customer_id', 'vendor', 'transaction_datetime']
).reset_index(drop=True)

df_transactions['next_transaction_date'] = df_transactions.groupby(
    ['customer_id', 'vendor']
)['transaction_datetime'].shift(-1)

df_transactions['likelihood_prediction'] = (
    df_transactions['next_transaction_date'] - df_transactions['transaction_datetime']
).dt.days

In [14]:
import os

os.makedirs("predictor/data", exist_ok=True)

In [15]:
df_transactions.to_csv("predictor/data/dataset1.csv", index=False)